# Fact Server Performance

## Overview
Creates **fact_server_performance** fact table containing server-level performance metrics.

**Source**: server_performance_data (silver)
**Dimensions**: dim_device, dim_time
**Target**: telecom_catalog.gold_schema.fact_server_performance

---

## Step 1: Create Fact Table
Define fact table schema with device_key, time_key (FKs), and server metrics (disk usage, network errors, uptime).

In [0]:
CREATE OR REPLACE TABLE telecom_catalog.gold_schema.fact_server_performance
(
    device_key BIGINT NOT NULL,
    time_key INT NOT NULL,

    disk_usage DOUBLE,
    network_errors DOUBLE,
    uptime_hours INT
)
USING DELTA;

---
## Step 2: Prepare Source View
Join server_performance_data with dim_device (on host_id) and dim_time (on log_time date).

In [0]:
CREATE OR REPLACE TEMP VIEW vw_fact_server_performance_source AS

SELECT

    dd.device_key,
    dt.time_key,

    sp.disk_usage,
    sp.network_errors,
    sp.uptime_hours

FROM telecom_catalog.silver_schema.server_performance_data sp

INNER JOIN telecom_catalog.gold_schema.dim_device dd
    ON sp.host_id = dd.host_id

INNER JOIN telecom_catalog.gold_schema.dim_time dt
    ON TO_DATE(sp.log_time) = dt.full_date;

---
## Step 3: Validate Source Count
Verify source view record count.

In [0]:
SELECT COUNT(*)
FROM vw_fact_server_performance_source;

---
## Step 4: Check Foreign Keys
Validate no NULL values in device_key or time_key.

In [0]:
SELECT

SUM(CASE WHEN device_key IS NULL THEN 1 ELSE 0 END) AS null_device_key,

SUM(CASE WHEN time_key IS NULL THEN 1 ELSE 0 END) AS null_time_key

FROM vw_fact_server_performance_source;

---
## Step 5: Load Fact Table
Insert all records from source view into fact table.

In [0]:
INSERT INTO telecom_catalog.gold_schema.fact_server_performance
SELECT *
FROM vw_fact_server_performance_source;

---
## Summary
**Table**: `telecom_catalog.gold_schema.fact_server_performance`

**Grain**: One row per device per day

**Metrics**: disk_usage, network_errors, uptime_hours

**Dimensions**: device_key (FK to dim_device), time_key (FK to dim_time)

**Load**: INSERT from server_performance_data (silver)